In [ ]:
import os
import pandas as pd
import create_graphs

In [ ]:
data = 'lung'
# data = 'aml'

In [ ]:
path = '/Output/mastro_nsclc/2026-10-29_15-54-09/'
max_sizes = path + 'graph_orders_' + data + '.csv'
sizes_df = pd.read_csv(max_sizes)
data_k =  'out_k'
sizes_df['Folder'] = sizes_df['Folder'].str.replace(data_k, '', regex=True).astype(int)
sizes_df

In [ ]:
df_mastro_full = pd.read_csv('//MASTRO/MASTRO/trees-' + data + '_final.txt', sep=';')
df_mastro =  df_mastro_full#[df_mastro_full['pval_ind'] < 0.02] # df_mastro_full.nsmallest(40, 'pval_ind') #
max(df_mastro['pval_ind'])

In [ ]:
min(df_mastro['pval_ind'])

In [ ]:
df_mastro

In [ ]:
df_mastro['edges_traj'] = df_mastro['edges_traj'].str.replace('[\[\]]', '', regex=True)
df_mastro['edges_traj'] = df_mastro['edges_traj'].str.replace('g', '0', regex=True)
df_mastro['edges_traj'] = df_mastro['edges_traj'].str.replace(',', ' ', regex=True)
df_mastro['edges_traj']

In [ ]:
mastro_trajectories = []
for i, entry in enumerate(df_mastro['edges_traj']):
    traj = create_graphs.get_graph_from_line(entry, i)
    mastro_trajectories.append(traj)

# -1 to substract root node
df_mastro['traj_size'] = [g.number_of_nodes()-1 for g in mastro_trajectories]


In [ ]:
mask = df_mastro.apply(
    lambda row: row['traj_size'] >= sizes_df.loc[sizes_df['Folder'] == row['traj_supp'], 'Graph Order'].iloc[0]
    if (sizes_df['Graph Order'] == row['traj_size']).any() else False,
    axis=1
)
mastro_maximum = df_mastro[mask]
mastro_maximum

In [ ]:
mastro_maximum_traj = []
for i, entry in enumerate(mastro_maximum['edges_traj']):
    traj = create_graphs.get_graph_from_line(entry, i)
    mastro_maximum_traj.append(traj)


In [ ]:
dataframes = []
for folder in sorted(os.listdir(path)):
    if folder.startswith(data_k):
        file = path + folder + '/significance_output.txt'
        df = pd.read_csv(file, sep=';', index_col=False)
        dataframes.append(df)
        
df_pottr_full = pd.concat(dataframes)
print(df_pottr_full.shape)
df_pottr = df_pottr_full#[df_pottr_full['pval_ind'] < 0.02] # df_pottr_full[df_pottr_full['pval_ind'] <= max(df_mastro['pval_ind'])] #
print(df_pottr.shape)

df_pottr['edges_traj'] = df_pottr['edges_traj'].str.replace('[\[\]]', '', regex=True)
df_pottr['edges_traj'] = df_pottr['edges_traj'].str.replace(',', ' ', regex=True)

pottr_results = df_pottr['edges_traj']
df_pottr_full
#[df_pottr_full['edges_traj'] == '0->-PIK3CAamp 0->-TP53 0->-SOX2amp 0->-NFE2L2 PIK3CAamp->-NFE2L2 TP53->-NFE2L2 SOX2amp->-NFE2L2']

In [ ]:
df_pottr_full[df_pottr_full['pval_ind'] > 0.05].drop_duplicates()

In [ ]:
df_pottr_full[df_pottr_full['pval_POTTR'] > 0.05].drop_duplicates()

In [ ]:
pottr_trajectories = []
for i, entry in enumerate(pottr_results):
    pottr_trajectories.append(create_graphs.get_graph_from_line(entry, i))
    if i == 32:
        g = create_graphs.get_graph_from_line(entry, i)
        print(g.nodes)

In [ ]:
def hash_graph(G):
    tuple_of_edges = tuple(sorted(G.edges, key=lambda x: (x[0], x[1])))
    tuple_of_nodes = tuple(sorted(G.nodes))
    return hash(tuple_of_edges + tuple_of_nodes)

In [ ]:
dict_graphs = dict()
for G in mastro_maximum_traj:
    graph_hash = hash_graph(G)
    dict_graphs[graph_hash] = G

In [ ]:
pottr_hashes = dict()
diff = dict()
same = dict()
too_small = dict()
for graph in pottr_trajectories:
    graph_hash = hash_graph(graph)
    pottr_hashes[graph_hash] = graph
    # > 2 otherwise just 1 alteration and root node
    if len(graph.nodes) > 2:
        if graph_hash not in dict_graphs:
            diff[graph_hash] = [graph.nodes, graph.edges]
        if graph_hash in dict_graphs:
            same[graph_hash] = [graph.nodes, graph.edges]
    else:
        too_small[graph_hash] = [graph.nodes, graph.edges]
            
print('unique pottr', len(diff))
for key in diff:
    print(diff[key])

In [ ]:
print('shared', len(same))

In [ ]:
print(len(too_small))

In [ ]:
for graph_hash in dict_graphs:
    graph = dict_graphs[graph_hash]
    if graph_hash not in pottr_hashes:
        print(graph.nodes)

In [ ]:
sizes = []
for h in pottr_hashes:
    graph = pottr_hashes[h]
    sizes.append(len(graph.nodes))

In [ ]:
print('unique mastro', len(dict_graphs) - len(same))

In [ ]:
sizes = set()
for h in dict_graphs:
    graph = dict_graphs[h]
    sizes.add(len(graph.nodes))
sizes

In [ ]:
len(pottr_hashes)-len(too_small)

In [ ]:
len(dict_graphs)